In [1]:
# import pandas as pd
# import cv2
# import os
# import pdb
# from tqdm import tqdm

# class videoReader(object):
#     def __init__(self, video_path, frame_interval=1, frame_kept_per_second=1):
#         self.video_path = video_path
#         self.frame_interval = frame_interval
#         self.frame_kept_per_second = frame_kept_per_second

#         #pdb.set_trace()
#         self.vid = cv2.VideoCapture(self.video_path)
#         self.fps = int(self.vid.get(cv2.CAP_PROP_FPS))
#         self.video_frames = self.vid.get(cv2.CAP_PROP_FRAME_COUNT)
#         self.video_len = int(self.video_frames/self.fps)


#     def video2frame(self, frame_save_path):
#         self.frame_save_path = frame_save_path
#         success, image = self.vid.read()
#         count = 0
#         while success:
#             count +=1
#             if count % self.frame_interval == 0:
#                 save_name = '{}/frame_{}_{}.jpg'.format(self.frame_save_path, int(count/self.fps), count)  # filename_second_index
#                 cv2.imencode('.jpg', image)[1].tofile(save_name)
#             success, image = self.vid.read()


#     def video2frame_update(self, frame_save_path):
#         self.frame_save_path = frame_save_path

#         count = 0
#         frame_interval = int(self.fps/self.frame_kept_per_second)
#         while(count < self.video_frames):
#             ret, image = self.vid.read()
#             if not ret:
#                 break
#             if count % self.fps == 0:
#                 frame_id = 0
#             if frame_id<frame_interval*self.frame_kept_per_second and frame_id%frame_interval == 0:
#                 save_name = '{0}/{1:05d}.jpg'.format(self.frame_save_path, count)
#                 cv2.imencode('.jpg', image)[1].tofile(save_name)

#             frame_id += 1
#             count += 1


# class CRAMED_dataset(object):
#     def __init__(self, path_to_dataset = './CREMA-D', frame_interval=1, frame_kept_per_second=1):
#         self.path_to_video = os.path.join(path_to_dataset, 'VideoFlash')
#         self.path_to_audio = os.path.join(path_to_dataset, 'AudioWAV')
#         self.frame_kept_per_second = frame_kept_per_second
#         self.path_to_save = os.path.join('./NewData/Image-{:02d}-FPS'.format(self.frame_kept_per_second))
#         if not os.path.exists(self.path_to_save):
#             os.mkdir(self.path_to_save)

#         csv_file = pd.read_csv(os.path.join(path_to_dataset, 'SentenceFilenames.csv'))
#         self.file_list = list(csv_file['Filename'])

#     def extractImage(self):

#         for each_video in tqdm(self.file_list, desc="Processing videos"):
#             # tqdm.write(f'Processing {each_video} ...')  # 使用 tqdm.write 避免进度条混乱
#             video_dir = os.path.join(self.path_to_video, each_video+'.flv')
#             self.videoReader = videoReader(video_path=video_dir, frame_kept_per_second=self.frame_kept_per_second)
        
#             save_dir = os.path.join(self.path_to_save, each_video)
#             if not os.path.exists(save_dir):
#                 os.mkdir(save_dir)
#             self.videoReader.video2frame_update(frame_save_path=save_dir)


# cramed = CRAMED_dataset()
# cramed.extractImage()

Processing videos:  16%|█▌        | 1179/7442 [01:18<06:54, 15.09it/s]


KeyboardInterrupt: 

In [3]:
import pandas as pd
import cv2
import os
import pdb
from tqdm import tqdm
from multiprocessing import Pool, cpu_count
import functools

class videoReader(object):
    def __init__(self, video_path, frame_interval=1, frame_kept_per_second=1):
        self.video_path = video_path
        self.frame_interval = frame_interval
        self.frame_kept_per_second = frame_kept_per_second

        #pdb.set_trace()
        self.vid = cv2.VideoCapture(self.video_path)
        self.fps = int(self.vid.get(cv2.CAP_PROP_FPS))
        self.video_frames = self.vid.get(cv2.CAP_PROP_FRAME_COUNT)
        self.video_len = int(self.video_frames/self.fps)


    def video2frame(self, frame_save_path):
        self.frame_save_path = frame_save_path
        success, image = self.vid.read()
        count = 0
        while success:
            count +=1
            if count % self.frame_interval == 0:
                save_name = '{}/frame_{}_{}.jpg'.format(self.frame_save_path, int(count/self.fps), count)  # filename_second_index
                cv2.imencode('.jpg', image)[1].tofile(save_name)
            success, image = self.vid.read()


    def video2frame_update(self, frame_save_path):
        self.frame_save_path = frame_save_path

        count = 0
        frame_interval = int(self.fps/self.frame_kept_per_second)
        while(count < self.video_frames):
            ret, image = self.vid.read()
            if not ret:
                break
            if count % self.fps == 0:
                frame_id = 0
            if frame_id<frame_interval*self.frame_kept_per_second and frame_id%frame_interval == 0:
                save_name = '{0}/{1:05d}.jpg'.format(self.frame_save_path, count)
                cv2.imencode('.jpg', image)[1].tofile(save_name)

            frame_id += 1
            count += 1


def process_single_video(each_video, path_to_video, path_to_save, frame_kept_per_second):
    """处理单个视频的函数，用于多进程调用"""
    try:
        video_dir = os.path.join(path_to_video, each_video + '.flv')
        video_reader = videoReader(video_path=video_dir, frame_kept_per_second=frame_kept_per_second)
        
        save_dir = os.path.join(path_to_save, each_video)
        if not os.path.exists(save_dir):
            os.mkdir(save_dir)
        video_reader.video2frame_update(frame_save_path=save_dir)
        
        # 释放视频资源
        video_reader.vid.release()
        return each_video, True, None
    except Exception as e:
        return each_video, False, str(e)


class CRAMED_dataset(object):
    def __init__(self, path_to_dataset='./CREMA-D', frame_interval=1, frame_kept_per_second=1):
        self.path_to_video = os.path.join(path_to_dataset, 'VideoFlash')
        self.path_to_audio = os.path.join(path_to_dataset, 'AudioWAV')
        self.frame_kept_per_second = frame_kept_per_second
        self.path_to_save = os.path.join('./NewData/Image-{:02d}-FPS'.format(self.frame_kept_per_second))
        if not os.path.exists(self.path_to_save):
            os.mkdir(self.path_to_save)

        csv_file = pd.read_csv(os.path.join(path_to_dataset, 'SentenceFilenames.csv'))
        self.file_list = list(csv_file['Filename'])

    def extractImage(self, num_processes=None):
        """使用多进程提取图像"""
        if num_processes is None:
            # num_processes = min(cpu_count(), len(self.file_list))  # 使用CPU核心数或文件数中较小的
            num_processes = 8  # 使用CPU核心数或文件数中较小的
        
        print(f"使用 {num_processes} 个进程处理 {len(self.file_list)} 个视频")
        
        # 创建部分应用的函数，固定部分参数
        process_func = functools.partial(
            process_single_video,
            path_to_video=self.path_to_video,
            path_to_save=self.path_to_save,
            frame_kept_per_second=self.frame_kept_per_second
        )
        
        # 使用多进程池
        with Pool(processes=num_processes) as pool:
            results = list(tqdm(
                pool.imap(process_func, self.file_list),
                total=len(self.file_list),
                desc="Processing videos",
                unit="video"
            ))
        
        # 检查处理结果
        success_count = 0
        for video_name, success, error in results:
            if success:
                success_count += 1
            else:
                print(f"处理视频 {video_name} 失败: {error}")
        
        print(f"处理完成: {success_count}/{len(self.file_list)} 个视频成功处理")


if __name__ == "__main__":
    # 在Windows上需要添加这个条件
    cramed = CRAMED_dataset()
    cramed.extractImage()

使用 8 个进程处理 7442 个视频


Processing videos: 100%|██████████| 7442/7442 [01:51<00:00, 66.85video/s]

处理完成: 7442/7442 个视频成功处理


In [4]:
import csv
import os
import numpy as np
import torchaudio
import torch
from multiprocessing import Pool, cpu_count
from tqdm import tqdm
import functools

class AudioProcessor:
    def __init__(self, audio_path, save_path, target_length=1024, norm_mean=-4.503877, norm_std=5.141276):
        self.audio_path = audio_path
        self.save_path = save_path
        self.target_length = target_length
        self.norm_mean = norm_mean
        self.norm_std = norm_std
        
        # 创建保存目录
        if not os.path.exists(self.save_path):
            os.makedirs(self.save_path)
    
    def process_single_audio(self, name):
        """处理单个音频文件"""
        try:
            # 检查文件是否存在
            if not os.path.exists(os.path.join(self.audio_path, name + '.wav')):
                return name, False, f"File {name}.wav not found"
            
            # 加载音频文件
            waveform, sr = torchaudio.load(os.path.join(self.audio_path, name + '.wav'))
            waveform = waveform - waveform.mean()
            
            # 提取fbank特征
            fbank = torchaudio.compliance.kaldi.fbank(
                waveform, 
                htk_compat=True, 
                sample_frequency=sr, 
                use_energy=False,
                window_type='hanning', 
                num_mel_bins=128, 
                dither=0.0, 
                frame_shift=10
            )
            
            # 调整长度
            n_frames = fbank.shape[0]
            p = self.target_length - n_frames
            
            # 裁剪或填充
            if p > 0:
                m = torch.nn.ZeroPad2d((0, 0, 0, p))
                fbank = m(fbank)
            elif p < 0:
                fbank = fbank[0:self.target_length, :]
            
            # 归一化
            fbank = (fbank - self.norm_mean) / (self.norm_std * 2)
            
            # 保存特征
            np.save(os.path.join(self.save_path, name + '.npy'), fbank)
            
            return name, True, None
            
        except Exception as e:
            return name, False, str(e)
    
    def process_audio_list(self, audio_list, num_processes=None, chunksize=1):
        """使用多进程处理音频列表"""
        if num_processes is None:
            num_processes = min(cpu_count(), len(audio_list))
        
        print(f"使用 {num_processes} 个进程处理 {len(audio_list)} 个音频文件")
        
        # 使用多进程池处理
        with Pool(processes=num_processes) as pool:
            results = list(tqdm(
                pool.imap(self.process_single_audio, audio_list, chunksize=chunksize),
                total=len(audio_list),
                desc="Processing audio files",
                unit="file"
            ))
        
        # 统计结果
        success_count = 0
        failed_files = []
        
        for name, success, error in results:
            if success:
                success_count += 1
            else:
                failed_files.append((name, error))
                print(f"处理音频 {name} 失败: {error}")
        
        print(f"处理完成: {success_count}/{len(audio_list)} 个音频文件成功处理")
        
        if failed_files:
            print(f"失败的文件列表:")
            for name, error in failed_files:
                print(f"  {name}: {error}")
        
        return success_count, failed_files


def load_audio_list(csv_file, audio_path):
    """从CSV文件加载音频列表"""
    data = []
    with open(csv_file) as f:
        for line in f:
            item = line.split("\n")[0].split(" ")
            name = item[0][:-4]  # 移除文件扩展名
            
            # 检查文件是否存在
            if os.path.exists(os.path.join(audio_path, name + '.wav')):
                data.append(name)
    
    return data


def main():
    # 参数设置
    save_path = './NewData/audio'  # 处理后的频谱图保存路径
    audio_path = './CREMA-D/AudioWAV'  # 音频文件路径
    csv_file = './CREMA-D/test.csv'  # 音频文件列表
    
    # 加载音频列表
    print("加载音频文件列表...")
    audio_list = load_audio_list(csv_file, audio_path)
    print(f"找到 {len(audio_list)} 个音频文件")
    
    # 创建音频处理器
    processor = AudioProcessor(audio_path, save_path)
    
    # 处理音频文件
    success_count, failed_files = processor.process_audio_list(
        audio_list, 
        num_processes=min(8, cpu_count()),  # 限制最大进程数，避免内存不足
        chunksize=5  # 调整块大小以优化性能
    )
    
    print(f"音频特征提取完成，成功处理 {success_count} 个文件")


if __name__ == "__main__":
    main()

加载音频文件列表...
找到 744 个音频文件
使用 8 个进程处理 744 个音频文件


Processing audio files: 100%|██████████| 744/744 [00:05<00:00, 133.06file/s]


处理完成: 744/744 个音频文件成功处理
音频特征提取完成，成功处理 744 个文件
